# LeetCode 1592 — Rearrange Spaces Between Words bằng Python

**Link đề:** https://leetcode.com/problems/rearrange-spaces-between-words/
**Độ khó:** Easy · **Chủ đề:** String

```python
class Solution:
    def reorderSpaces(self, text: str) -> str:
        ...
```

## Mục tiêu học
1. Nắm các công cụ chuỗi cơ bản: `str.count`, `str.split()` (và khác biệt với `split(" ")`), `" ".join`, nhân chuỗi `" " * k`.
2. Hiểu **chia nguyên và chia lấy dư** (`//`, `%`, `divmod`) qua bài "chia đều khoảng trắng".
3. Xử lý đúng **trường hợp đặc biệt chỉ có 1 từ** (không có khe nào để chia → tránh chia cho 0).
4. Viết 3 lời giải: dùng hàm có sẵn → tự quét chuỗi không dùng `split` → dựng kết quả bằng list ký tự; trace và test đối chiếu.
5. Nhận dạng pattern "phân phối đều + phần dư" (gặp lại ở Text Justification, chia kẹo, chia trang...).

## 1. Đề bài

Cho chuỗi `text` gồm các **từ** (chữ thường a–z) xen giữa các **khoảng trắng**. Hai từ cách nhau ít nhất một khoảng trắng; `text` có ít nhất một từ.

Hãy sắp xếp lại khoảng trắng sao cho:
- Số khoảng trắng giữa **mọi cặp từ kề nhau bằng nhau** và **lớn nhất có thể**.
- Nếu không chia đều được, phần **dư** đặt ở **cuối** chuỗi.
- Chuỗi trả về có **cùng độ dài** với `text`.

| `text` | Kết quả | Giải thích |
|---|---|---|
| `"  this   is  a sentence "` | `"this   is   a   sentence"` | 9 khoảng trắng, 4 từ → 3 khe, mỗi khe 3, dư 0 |
| `" practice   makes   perfect"` | `"practice   makes   perfect "` | 7 khoảng trắng, 3 từ → 2 khe, mỗi khe 3, dư 1 ở cuối |

**Constraints:** `1 <= text.length <= 100`; chỉ gồm chữ thường và `' '`; có ít nhất một từ.

Câu hỏi then chốt: có bao nhiêu khoảng trắng, bao nhiêu **khe** giữa các từ, và chia thế nào?

In [1]:
examples = [
    ("  this   is  a sentence ", "this   is   a   sentence"),
    (" practice   makes   perfect", "practice   makes   perfect "),
]
for text, expected in examples:
    print(repr(text), "->", repr(expected), "| độ dài", len(text), len(expected))

'  this   is  a sentence ' -> 'this   is   a   sentence' | độ dài 24 24
' practice   makes   perfect' -> 'practice   makes   perfect ' | độ dài 27 27


Mẹo quan sát: dùng `repr(s)` để in chuỗi **kèm dấu nháy**, nhờ đó thấy rõ khoảng trắng ở đầu/cuối. In thường thì khoảng trắng cuối "vô hình".

In [2]:
s = "abc  "
print(s, "|")        # khó thấy 2 dấu cách cuối
print(repr(s))       # 'abc  '
print(s.replace(" ", "·"))   # thay dấu cách bằng chấm giữa để nhìn

abc   |
'abc  '
abc··


## 2. Nền tảng Python cho chuỗi

### 2.1. `str.count(" ")` — đếm khoảng trắng

In [3]:
text = "  this   is  a sentence "
print("số khoảng trắng:", text.count(" "))
print("số chữ 's':", text.count("s"))

số khoảng trắng: 9
số chữ 's': 3


### 2.2. `split()` khác `split(" ")` — điểm cực kỳ dễ nhầm

- `text.split()` (**không tham số**): tách theo **mọi cụm** khoảng trắng, **bỏ** khoảng trắng đầu/cuối → chỉ còn các từ.
- `text.split(" ")`: tách tại **từng** dấu cách → sinh ra nhiều chuỗi rỗng `''`.

Bài này cần danh sách từ → dùng `split()`.

In [4]:
text = "  this   is  a sentence "
print("split()    ->", text.split())
print("split(' ') ->", text.split(" "))

split()    -> ['this', 'is', 'a', 'sentence']
split(' ') -> ['', '', 'this', '', '', 'is', '', 'a', 'sentence', '']


### 2.3. `" ".join(words)` và nhân chuỗi `" " * k`

- `sep.join(list_of_str)` nối các chuỗi, chèn `sep` vào **giữa** (không chèn ở đầu/cuối).
- `" " * 3` cho `"   "` (3 dấu cách). `" " * 0` cho chuỗi rỗng `""`.

In [5]:
words = ["this", "is", "a", "sentence"]
print(repr("-".join(words)))
gap = " " * 3
print(repr(gap.join(words)))
print(repr(" " * 0), "| độ dài", len(" " * 0))
print(repr("".join(["a", "b"])))

'this-is-a-sentence'
'this   is   a   sentence'
'' | độ dài 0
'ab'


### 2.4. `//`, `%` và `divmod` — chia đều và phần dư

Có `spaces` khoảng trắng, chia cho `gaps` khe:
- Mỗi khe: `spaces // gaps` (chia nguyên).
- Dư: `spaces % gaps`.
- `divmod(a, b)` trả về **cả hai** cùng lúc: `(a // b, a % b)`.

Luôn có: `gaps * (spaces // gaps) + spaces % gaps == spaces`.

In [6]:
spaces, gaps = 7, 2
print("mỗi khe:", spaces // gaps, "| dư:", spaces % gaps)
q, r = divmod(spaces, gaps)
print("divmod ->", (q, r), "| kiểm tra:", gaps * q + r == spaces)
for spaces, gaps in [(9, 3), (7, 2), (5, 4), (2, 5)]:
    print(f"spaces={spaces}, gaps={gaps} -> divmod={divmod(spaces, gaps)}")

mỗi khe: 3 | dư: 1
divmod -> (3, 1) | kiểm tra: True
spaces=9, gaps=3 -> divmod=(3, 0)
spaces=7, gaps=2 -> divmod=(3, 1)
spaces=5, gaps=4 -> divmod=(1, 1)
spaces=2, gaps=5 -> divmod=(0, 2)


### 2.5. Chia cho 0 — vì sao phải cẩn thận khi chỉ có 1 từ

Nếu `text` chỉ có **1 từ**, số khe là `1 - 1 = 0`. Phép `spaces // 0` sẽ báo lỗi `ZeroDivisionError`. Khi đó đề yêu cầu: **toàn bộ** khoảng trắng dồn về cuối.

In [7]:
try:
    print(5 // 0)
except ZeroDivisionError as e:
    print("ZeroDivisionError:", e)

ZeroDivisionError: integer division or modulo by zero


## 3. Cách nghĩ trực giác — làm bằng tay

Với `" practice   makes   perfect"`:

1. Đếm khoảng trắng: `1 + 3 + 3 = 7`.
2. Lấy danh sách từ: `["practice", "makes", "perfect"]` → 3 từ → **2 khe**.
3. Chia: `7 // 2 = 3` mỗi khe, dư `7 % 2 = 1`.
4. Ghép: `"practice" + 3 cách + "makes" + 3 cách + "perfect" + 1 cách`.

Kiểm tra độ dài: `8 + 5 + 7` chữ + `7` cách = `27` = độ dài gốc. ✅

In [8]:
text = " practice   makes   perfect"
spaces = text.count(" ")
words = text.split()
gaps = len(words) - 1
gap, extra = divmod(spaces, gaps)
print("spaces =", spaces, "| words =", words, "| gaps =", gaps)
print("mỗi khe =", gap, "| dư =", extra)
result = (" " * gap).join(words) + " " * extra
print(repr(result), "| độ dài", len(result), "==", len(text))

spaces = 7 | words = ['practice', 'makes', 'perfect'] | gaps = 2
mỗi khe = 3 | dư = 1
'practice   makes   perfect ' | độ dài 27 == 27


## 4. Nút thắt và insight

Không có "nút thắt" về tốc độ (chuỗi ≤ 100 ký tự). Nút thắt thật sự là **tính đúng**:

- **Khe** chứ không phải **từ**: chia cho `len(words) - 1`, không phải `len(words)`.
- **1 từ** → 0 khe → không chia được: mọi khoảng trắng ra cuối.
- Phần dư chỉ đặt ở **cuối**, không rải lên đầu.

**Insight gọn:** kết quả = `từ₁ + G + từ₂ + G + ... + từ_k + R`, với `G = " " * (S // (k-1))`, `R = " " * (S % (k-1))`; riêng `k = 1`: `từ₁ + " " * S`.

Cách thống nhất hai trường hợp: nếu `k == 1` thì đặt `gap = 0, extra = S` — vì `"".join([w])` chỉ là `w`, công thức chung vẫn đúng.

In [9]:
print(repr(("   ").join(["hello"])))   # join một phần tử -> không chèn gì
text = "  hello "
S, words = text.count(" "), text.split()
gap, extra = (0, S) if len(words) == 1 else divmod(S, len(words) - 1)
print(repr((" " * gap).join(words) + " " * extra))

'hello'
'hello   '


Ở trên có **biểu thức điều kiện** (ternary): `A if điều_kiện else B` — trả `A` khi điều kiện đúng, ngược lại trả `B`. Kết hợp với **gán nhiều biến** `gap, extra = (0, S)`.

In [10]:
k = 1
label = "một từ" if k == 1 else "nhiều từ"
x, y = (10, 20) if k == 1 else (1, 2)
print(label, x, y)

một từ 10 20


## 5. Lời giải chính (Cách 1: split + divmod + join) — giảng từng đoạn

### 5.1. Đoạn 1: đếm khoảng trắng và tách từ

In [11]:
text = "  this   is  a sentence "
spaces = text.count(" ")
words = text.split()
print(spaces, words)

9 ['this', 'is', 'a', 'sentence']


### 5.2. Đoạn 2: tính số khoảng trắng mỗi khe và phần dư (có xử lý 1 từ)

In [12]:
if len(words) == 1:
    gap, extra = 0, spaces
else:
    gap, extra = divmod(spaces, len(words) - 1)
print("gap =", gap, "| extra =", extra)

gap = 3 | extra = 0


### 5.3. Đoạn 3: ghép kết quả

In [13]:
result = (" " * gap).join(words) + " " * extra
print(repr(result))
assert len(result) == len(text)

'this   is   a   sentence'


### 5.4. Ghép thành hàm

In [14]:
def reorder_spaces_split(text: str) -> str:
    spaces = text.count(" ")
    words = text.split()
    if len(words) == 1:
        gap, extra = 0, spaces
    else:
        gap, extra = divmod(spaces, len(words) - 1)
    return (" " * gap).join(words) + " " * extra

for text, expected in examples:
    out = reorder_spaces_split(text)
    print(repr(out), out == expected)

'this   is   a   sentence' True
'practice   makes   perfect ' True


## 6. Cách 2 — Tự quét chuỗi, không dùng `split`

Mục đích: hiểu `split()` làm gì bên trong. Duyệt từng ký tự:
- Gặp dấu cách: tăng `spaces`; nếu đang gom dở một từ thì "đóng" từ đó lại.
- Gặp chữ: thêm vào từ hiện tại `current`.
- Cuối chuỗi: nếu `current` còn dở thì đóng lại.

Đây là mẫu **"gom cụm liên tiếp"** (tokenize) rất hay gặp.

In [15]:
def tokenize(text: str):
    words, current, spaces = [], "", 0
    for ch in text:
        if ch == " ":
            spaces += 1
            if current:              # chuỗi khác rỗng -> True
                words.append(current)
                current = ""
        else:
            current += ch
    if current:                      # đừng quên từ cuối!
        words.append(current)
    return words, spaces

print(tokenize("  this   is  a sentence "))
print(tokenize(" practice   makes   perfect"))   # từ cuối không có cách theo sau

(['this', 'is', 'a', 'sentence'], 9)
(['practice', 'makes', 'perfect'], 7)


Lưu ý: `if current:` — trong Python chuỗi rỗng `""` được coi là **False**, chuỗi có ký tự là **True** (gọi là *truthiness*).

In [16]:
for s in ["", "a", " "]:
    print(repr(s), "->", bool(s))

'' -> False
'a' -> True
' ' -> True


In [17]:
def reorder_spaces_scan(text: str) -> str:
    words, spaces = tokenize(text)
    k = len(words)
    gap = spaces // (k - 1) if k > 1 else 0
    extra = spaces - gap * (k - 1)
    out = ""
    for idx, w in enumerate(words):
        out += w
        if idx < k - 1:
            out += " " * gap
    return out + " " * extra

print([reorder_spaces_scan(t) == e for t, e in examples])

[True, True]


Chú ý cách tính `extra = spaces - gap * (k - 1)`: không cần `%`, và công thức vẫn đúng khi `k = 1` (`gap = 0` → `extra = spaces`).

## 7. Cách 3 — Dựng kết quả bằng list ký tự

Nối chuỗi bằng `+=` trong vòng lặp có thể tạo nhiều chuỗi tạm (chuỗi Python **bất biến** — immutable). Cách chuẩn là gom các mảnh vào **list** rồi `"".join(list)` một lần ở cuối.

In [18]:
def reorder_spaces_list(text: str) -> str:
    words = text.split()
    spaces = len(text) - sum(len(w) for w in words)   # cách khác để đếm khoảng trắng
    k = len(words)
    gap = spaces // (k - 1) if k > 1 else 0
    parts = []
    for idx, w in enumerate(words):
        if idx > 0:
            parts.append(" " * gap)
        parts.append(w)
    parts.append(" " * (spaces - gap * (k - 1)))
    return "".join(parts)

print([reorder_spaces_list(t) == e for t, e in examples])

[True, True]


Mẹo ở cách 3: số khoảng trắng = `len(text) - tổng độ dài các từ` (vì chuỗi chỉ gồm chữ và dấu cách). `sum(len(w) for w in words)` là **generator expression**: tính độ dài từng từ rồi cộng.

In [19]:
words = ["this", "is", "a", "sentence"]
print([len(w) for w in words], "-> tổng", sum(len(w) for w in words))
print("khoảng trắng =", len("  this   is  a sentence ") - sum(len(w) for w in words))

[4, 2, 1, 8] -> tổng 15
khoảng trắng = 9


## 8. TRACE từng bước

Trace với `"  hello   world  ok "`:

| Bước | Giá trị |
|---|---|
| Đếm khoảng trắng | `2 + 3 + 2 + 1 = 8` |
| Từ | `["hello", "world", "ok"]`, `k = 3` |
| Số khe | `k - 1 = 2` |
| `divmod(8, 2)` | `(4, 0)` → mỗi khe 4, dư 0 |
| Kết quả | `"hello····world····ok"` |

Và trace quá trình tokenize (cách 2) để thấy lúc nào một từ được "đóng".

In [20]:
def trace_reorder(text: str) -> str:
    print("text =", repr(text), "| len =", len(text))
    words, current, spaces = [], "", 0
    for i, ch in enumerate(text):
        if ch == " ":
            spaces += 1
            action = f"cách -> spaces={spaces}"
            if current:
                words.append(current)
                action += f", đóng từ {current!r}"
                current = ""
        else:
            current += ch
            action = f"chữ -> current={current!r}"
        print(f"  i={i:>2} ch={ch!r}: {action}")
    if current:
        words.append(current)
        print(f"  hết chuỗi: đóng từ {current!r}")
    k = len(words)
    gap, extra = (divmod(spaces, k - 1) if k > 1 else (0, spaces))
    print(f"words={words}, k={k}, spaces={spaces}, gap={gap}, extra={extra}")
    out = (" " * gap).join(words) + " " * extra
    print("kết quả =", repr(out))
    return out

trace_reorder("  hello   world  ok ")

text = '  hello   world  ok ' | len = 20
  i= 0 ch=' ': cách -> spaces=1
  i= 1 ch=' ': cách -> spaces=2
  i= 2 ch='h': chữ -> current='h'
  i= 3 ch='e': chữ -> current='he'
  i= 4 ch='l': chữ -> current='hel'
  i= 5 ch='l': chữ -> current='hell'
  i= 6 ch='o': chữ -> current='hello'
  i= 7 ch=' ': cách -> spaces=3, đóng từ 'hello'
  i= 8 ch=' ': cách -> spaces=4
  i= 9 ch=' ': cách -> spaces=5
  i=10 ch='w': chữ -> current='w'
  i=11 ch='o': chữ -> current='wo'
  i=12 ch='r': chữ -> current='wor'
  i=13 ch='l': chữ -> current='worl'
  i=14 ch='d': chữ -> current='world'
  i=15 ch=' ': cách -> spaces=6, đóng từ 'world'
  i=16 ch=' ': cách -> spaces=7
  i=17 ch='o': chữ -> current='o'
  i=18 ch='k': chữ -> current='ok'
  i=19 ch=' ': cách -> spaces=8, đóng từ 'ok'
words=['hello', 'world', 'ok'], k=3, spaces=8, gap=4, extra=0
kết quả = 'hello    world    ok'


'hello    world    ok'

Trace trường hợp 1 từ — nhánh đặc biệt:

In [21]:
trace_reorder(" a  ")

text = ' a  ' | len = 4
  i= 0 ch=' ': cách -> spaces=1
  i= 1 ch='a': chữ -> current='a'
  i= 2 ch=' ': cách -> spaces=2, đóng từ 'a'
  i= 3 ch=' ': cách -> spaces=3
words=['a'], k=1, spaces=3, gap=0, extra=3
kết quả = 'a   '


'a   '

## 9. Code nộp LeetCode

### 9.1. Cách chính (split + divmod + join)

In [22]:
class Solution:
    def reorderSpaces(self, text: str) -> str:
        spaces = text.count(" ")
        words = text.split()
        if len(words) == 1:
            return words[0] + " " * spaces
        gap, extra = divmod(spaces, len(words) - 1)
        return (" " * gap).join(words) + " " * extra

sol = Solution()
for text, expected in examples:
    assert sol.reorderSpaces(text) == expected
print("Solution PASS ví dụ đề")

Solution PASS ví dụ đề


### 9.2. Cách tự quét (không dùng split)

In [23]:
class SolutionScan:
    def reorderSpaces(self, text: str) -> str:
        words, cur, spaces = [], [], 0
        for ch in text:
            if ch == " ":
                spaces += 1
                if cur:
                    words.append("".join(cur))
                    cur = []
            else:
                cur.append(ch)
        if cur:
            words.append("".join(cur))
        k = len(words)
        gap = spaces // (k - 1) if k > 1 else 0
        return (" " * gap).join(words) + " " * (spaces - gap * (k - 1))

print([SolutionScan().reorderSpaces(t) == e for t, e in examples])

[True, True]


### 9.3. Cách dựng list ký tự

In [24]:
class SolutionList:
    def reorderSpaces(self, text: str) -> str:
        words = text.split()
        spaces = len(text) - sum(map(len, words))
        k = len(words)
        gap = spaces // (k - 1) if k > 1 else 0
        parts = []
        for i, w in enumerate(words):
            if i:
                parts.append(" " * gap)
            parts.append(w)
        parts.append(" " * (spaces - gap * (k - 1)))
        return "".join(parts)

print([SolutionList().reorderSpaces(t) == e for t, e in examples])

[True, True]


`map(len, words)` áp dụng hàm `len` lên từng phần tử — tương đương `(len(w) for w in words)`.

## 10. Vì sao đúng? (invariant)

Gọi `S` = số khoảng trắng, `k` = số từ.

- **Bảo toàn ký tự:** kết quả chứa đúng các từ theo thứ tự gốc, cộng `gap·(k-1) + extra = S` khoảng trắng (tính chất của phép chia nguyên: `(k-1)·(S//(k-1)) + S%(k-1) = S`). Vậy độ dài bằng `len(text)`.
- **Đều nhau:** mọi khe đều dùng cùng chuỗi `" " * gap`.
- **Lớn nhất:** nếu mỗi khe có `gap + 1` thì cần `(k-1)(gap+1) > S` khoảng trắng (vì `extra < k-1`) → không đủ. Nên `gap = S // (k-1)` là lớn nhất.
- **Dư ở cuối:** `extra` được nối sau từ cuối.
- `k = 1`: không có khe, toàn bộ `S` ra cuối — đúng yêu cầu.

### 10.1. Độ phức tạp

Gọi `n = len(text)` (≤ 100).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. split + divmod + join | `O(n)` | `O(n)` | ngắn gọn nhất |
| 2. Tự quét | `O(n)` | `O(n)` | hiểu bản chất `split`; bản dùng `+=` chuỗi có thể tốn hơn |
| 3. Dựng list | `O(n)` | `O(n)` | mẫu "gom mảnh rồi join" chuẩn |

Mỗi cách đọc chuỗi một số lần hằng số, và kết quả dài `n` nên không thể dưới `O(n)`.

## 11. Bộ test

Nhóm test:
1. Ví dụ đề.
2. Một từ: không cách, cách đầu, cách cuối, cách hai bên.
3. Không có khoảng trắng dư thừa (đã chuẩn): `"a b"`.
4. Dư nhiều: số khe lớn hơn số khoảng trắng mỗi khe.
5. Kiểm tra **tính chất** trên chuỗi ngẫu nhiên: độ dài giữ nguyên, các từ giữ thứ tự, các khe bằng nhau, dư < số khe.

In [25]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<26} PASS {actual!r}")

solvers = [reorder_spaces_split, reorder_spaces_scan, reorder_spaces_list,
           Solution().reorderSpaces, SolutionScan().reorderSpaces, SolutionList().reorderSpaces]
cases = [
    ("ví dụ 1", "  this   is  a sentence ", "this   is   a   sentence"),
    ("ví dụ 2", " practice   makes   perfect", "practice   makes   perfect "),
    ("1 từ, không cách", "a", "a"),
    ("1 từ, cách đầu", "  hello", "hello  "),
    ("1 từ, hai bên", " hi ", "hi  "),
    ("đã chuẩn", "a b", "a b"),
    ("dư 2, 4 khe", "a b c d e  ", "a b c d e  "),
    ("5 cách, 4 khe", "a  b c d e", "a b c d e "),
    ("hai từ", "  x  y ", "x     y"),
]
for name, text, expected in cases:
    for f in solvers:
        assert f(text) == expected, (name, f, f(text))
    check(name, reorder_spaces_split(text), expected)

ví dụ 1                    PASS 'this   is   a   sentence'
ví dụ 2                    PASS 'practice   makes   perfect '
1 từ, không cách           PASS 'a'
1 từ, cách đầu             PASS 'hello  '
1 từ, hai bên              PASS 'hi  '
đã chuẩn                   PASS 'a b'
dư 2, 4 khe                PASS 'a b c d e  '
5 cách, 4 khe              PASS 'a b c d e '
hai từ                     PASS 'x     y'


In [26]:
import random, re
rng = random.Random(1592)

def random_text(rng):
    k = rng.randint(1, 6)
    words = ["".join(rng.choice("abcxyz") for _ in range(rng.randint(1, 4))) for _ in range(k)]
    seps = [" " * rng.randint(1, 4) for _ in range(k - 1)]
    t = " " * rng.randint(0, 3)
    for i, w in enumerate(words):
        t += w + (seps[i] if i < k - 1 else "")
    return t + " " * rng.randint(0, 3)

for _ in range(500):
    text = random_text(rng)
    out = reorder_spaces_split(text)
    for f in solvers:
        assert f(text) == out
    assert len(out) == len(text)                       # cùng độ dài
    assert out.split() == text.split()                 # cùng các từ, cùng thứ tự
    assert not out.startswith(" ")                     # không cách ở đầu
    words = text.split()
    if len(words) > 1:
        inner = out.rstrip(" ")
        # các khe bằng nhau
        gap_lens = {len(g) for g in re.findall(r" +", inner)}
        assert len(gap_lens) == 1
        assert len(out) - len(inner) < len(words) - 1  # phần dư < số khe
print("500 chuỗi ngẫu nhiên: 6 hàm khớp nhau và mọi tính chất đều đúng")

500 chuỗi ngẫu nhiên: 6 hàm khớp nhau và mọi tính chất đều đúng


Cell trên dùng `re.findall(r" +", s)` — **biểu thức chính quy** (regular expression, thư viện chuẩn `re`): `" +"` nghĩa là "một hoặc nhiều dấu cách liên tiếp". `findall` trả về list các cụm khớp. `rstrip(" ")` bỏ dấu cách ở **cuối** chuỗi.

In [27]:
import re
s = "this   is   a   sentence  "
print(re.findall(r" +", s))
print(repr(s.rstrip(" ")), "| dư cuối:", len(s) - len(s.rstrip(" ")))

['   ', '   ', '   ', '  ']
'this   is   a   sentence' | dư cuối: 2


## 12. Lỗi người mới hay gặp

### Lỗi 1: Dùng `split(" ")` thay vì `split()`

`split(" ")` trả về cả chuỗi rỗng, số "từ" bị đếm sai: 9 dấu cách tách ra 10 "mảnh" → 9 khe, mỗi khe 1 → ghép lại **y nguyên chuỗi gốc**, chẳng sắp xếp gì cả.

In [28]:
def wrong_split_space(text):
    spaces = text.count(" ")
    words = text.split(" ")             # SAI
    gap, extra = divmod(spaces, len(words) - 1)
    return (" " * gap).join(words) + " " * extra

t = "  this   is  a sentence "
print("mảnh:", t.split(" "))
print("sai :", repr(wrong_split_space(t)), "| giống input?", wrong_split_space(t) == t)
print("đúng:", repr(reorder_spaces_split(t)))

mảnh: ['', '', 'this', '', '', 'is', '', 'a', 'sentence', '']
sai : '  this   is  a sentence ' | giống input? True
đúng: 'this   is   a   sentence'


### Lỗi 2: Không xử lý 1 từ → `ZeroDivisionError`

In [29]:
def wrong_no_single(text):
    spaces, words = text.count(" "), text.split()
    gap, extra = divmod(spaces, len(words) - 1)
    return (" " * gap).join(words) + " " * extra

try:
    wrong_no_single(" hello ")
except ZeroDivisionError as e:
    print("ZeroDivisionError:", e, "-> phải tách riêng trường hợp 1 từ")

ZeroDivisionError: integer division or modulo by zero -> phải tách riêng trường hợp 1 từ


### Lỗi 3: Chia cho số từ thay vì số khe

`divmod(spaces, len(words))` coi như có `k` khe; kết quả có ít khoảng trắng giữa hơn và dư nhiều hơn mức cần.

In [30]:
def wrong_divide_by_words(text):
    spaces, words = text.count(" "), text.split()
    gap, extra = divmod(spaces, len(words))
    return (" " * gap).join(words) + " " * extra

t = "  this   is  a sentence "
print("sai :", repr(wrong_divide_by_words(t)))
print("đúng:", repr(reorder_spaces_split(t)))

sai : 'this  is  a  sentence '
đúng: 'this   is   a   sentence'


### Lỗi 4: Quên từ cuối khi tự quét

Nếu chuỗi kết thúc bằng chữ (không có dấu cách sau), vòng lặp không bao giờ "đóng" từ cuối. Phải có `if current: words.append(current)` sau vòng lặp.

In [31]:
def wrong_tokenize(text):
    words, current = [], ""
    for ch in text:
        if ch == " ":
            if current:
                words.append(current)
                current = ""
        else:
            current += ch
    return words                          # thiếu bước đóng từ cuối

print("sai :", wrong_tokenize(" practice   makes   perfect"))
print("đúng:", tokenize(" practice   makes   perfect")[0])

sai : ['practice', 'makes']
đúng: ['practice', 'makes', 'perfect']


## 13. Mẹo nhận dạng pattern

- "Chia **đều**, phần **dư** đặt ở đâu đó" → `divmod(tổng, số_phần)`; nhớ đếm **số khe = số phần tử − 1** khi chia khoảng giữa.
- Mọi phép chia: tự hỏi "**mẫu số có thể bằng 0 không?**".
- Chuỗi gồm từ + khoảng trắng: `split()` để lấy từ, `count(" ")` hoặc `len - tổng độ dài từ` để lấy khoảng trắng.
- Dựng chuỗi kết quả: gom vào list rồi `"".join`.
- Bài liên quan: 68 Text Justification (Hard — chính là bài này lặp trên từng dòng, phần dư rải từ trái), 151 Reverse Words in a String, 1816 Truncate Sentence, 2125 Number of Laser Beams (đếm theo nhóm).

## 14. Biến thể

### 15.1. Rải phần dư từ **trái** sang (kiểu Text Justification)

Thay vì dồn dư về cuối, `extra` khe đầu tiên mỗi khe thêm 1 khoảng trắng.

In [32]:
def justify_left_extra(text):
    spaces, words = text.count(" "), text.split()
    if len(words) == 1:
        return words[0] + " " * spaces
    gap, extra = divmod(spaces, len(words) - 1)
    parts = [words[0]]
    for i, w in enumerate(words[1:]):
        parts.append(" " * (gap + (1 if i < extra else 0)))
        parts.append(w)
    return "".join(parts)

print(repr(justify_left_extra(" practice   makes   perfect")))   # dư 1 vào khe đầu
print(repr(justify_left_extra("a  b c d e")))                    # 5 cách, 4 khe -> 2,1,1,1

'practice    makes   perfect'
'a  b c d e'


### 15.2. Căn giữa: chia đều khoảng trắng hai bên, phần dư ở phải

In [33]:
def center_words(text):
    spaces, words = text.count(" "), text.split()
    body = " ".join(words)                      # mỗi khe đúng 1 cách
    rest = spaces - (len(words) - 1)
    left, right = rest // 2, rest - rest // 2
    return " " * left + body + " " * right

print(repr(center_words("  hello   world ")))

'  hello world   '


### 15.3. Chuẩn hóa khoảng trắng: mỗi khe đúng 1, bỏ đầu/cuối

In [34]:
def normalize_spaces(text):
    return " ".join(text.split())

print(repr(normalize_spaces("  this   is  a sentence ")))

'this is a sentence'


## 15. Bài tập tự luyện (TODO)

**Bài 1 — tính tay.** Với `text = "  a  bb   c"`: có bao nhiêu khoảng trắng, bao nhiêu khe, `gap`, `extra`, và kết quả?

**Bài 2 — đoán output.** `"x y  z".split(" ")` và `"x y  z".split()` lần lượt in ra gì?

### Bài 3 — Điền TODO cho lời giải chính

In [35]:
def exercise_reorder(text: str) -> str:
    # TODO 1: spaces = số dấu cách; words = danh sách từ
    # TODO 2: nếu chỉ 1 từ -> trả từ đó + toàn bộ dấu cách
    # TODO 3: gap, extra = divmod(spaces, số khe)
    # TODO 4: ghép bằng join và thêm phần dư ở cuối
    pass

# assert exercise_reorder(" practice   makes   perfect") == "practice   makes   perfect "

### Bài 4 — Tự viết hàm tách từ không dùng `split`

In [36]:
def exercise_tokenize(text: str) -> list[str]:
    # TODO: gom cụm chữ liên tiếp, đừng quên từ cuối
    pass

# assert exercise_tokenize("  ab c  d") == ["ab", "c", "d"]

### Bài 5 — Biến thể: phần dư rải từ trái sang

In [37]:
def exercise_left_extra(text: str) -> str:
    # TODO: khe thứ i (0-based) nhận gap + 1 nếu i < extra
    pass

# assert exercise_left_extra("a  b c d e") == "a  b c d e"

### Bài 6 — Mini challenge: căn đều theo độ rộng cho trước

Viết `exercise_full_justify(words, width)` trả về một dòng dài đúng `width` chứa các từ, khoảng trắng chia đều giữa các từ, phần dư rải từ trái (nếu chỉ 1 từ thì dồn phải). Giả sử tổng độ dài từ + (k-1) ≤ width.

In [38]:
def exercise_full_justify(words: list[str], width: int) -> str:
    # TODO
    pass

# assert exercise_full_justify(["This", "is", "an"], 16) == "This    is    an"
# assert exercise_full_justify(["example", "of", "text"], 16) == "example  of text"

## 16. Gợi ý 3 mức

**Bài 1:** M1: đếm dấu cách trong từng cụm. M2: 3 từ → 2 khe. M3: `divmod(7, 2)`.

**Bài 2:** M1: `split(" ")` tách tại **từng** dấu cách. M2: giữa hai dấu cách liền nhau có một chuỗi rỗng. M3: kết quả thứ nhất có 4 phần tử.

**Bài 3:** M1: làm theo 4 bước của mục 5. M2: `if len(words) == 1: return words[0] + " " * spaces`. M3: `return (" " * gap).join(words) + " " * extra`.

**Bài 4:** M1: biến `current` đang gom. M2: gặp `" "` thì đóng `current` nếu khác rỗng. M3: sau vòng lặp `if current: words.append(current)`.

**Bài 5:** M1: xem biến thể 15.1. M2: duyệt `words[1:]` với chỉ số khe `i`. M3: `" " * (gap + (i < extra))` — `True` cộng như 1.

**Bài 6:** M1: `spaces = width - tổng độ dài từ`. M2: giống Bài 5. M3: trường hợp 1 từ: `w + " " * (width - len(w))`.

## 17. Đáp án tham khảo

**Bài 1:** khoảng trắng `2 + 2 + 3 = 7`, từ `["a", "bb", "c"]`, khe `2`, `gap = 3`, `extra = 1` → `"a   bb   c "`.

**Bài 2:** `['x', 'y', '', 'z']` và `['x', 'y', 'z']`.

In [39]:
def answer_reorder(text):
    spaces, words = text.count(" "), text.split()
    if len(words) == 1:
        return words[0] + " " * spaces
    gap, extra = divmod(spaces, len(words) - 1)
    return (" " * gap).join(words) + " " * extra

def answer_tokenize(text):
    words, cur = [], ""
    for ch in text:
        if ch == " ":
            if cur:
                words.append(cur)
                cur = ""
        else:
            cur += ch
    if cur:
        words.append(cur)
    return words

def answer_left_extra(text):
    spaces, words = text.count(" "), text.split()
    if len(words) == 1:
        return words[0] + " " * spaces
    gap, extra = divmod(spaces, len(words) - 1)
    out = words[0]
    for i, w in enumerate(words[1:]):
        out += " " * (gap + (i < extra)) + w
    return out

def answer_full_justify(words, width):
    if len(words) == 1:
        return words[0] + " " * (width - len(words[0]))
    spaces = width - sum(len(w) for w in words)
    gap, extra = divmod(spaces, len(words) - 1)
    out = words[0]
    for i, w in enumerate(words[1:]):
        out += " " * (gap + (i < extra)) + w
    return out

assert answer_reorder("  a  bb   c") == "a   bb   c "
assert "x y  z".split(" ") == ["x", "y", "", "z"] and "x y  z".split() == ["x", "y", "z"]
assert answer_reorder(" practice   makes   perfect") == "practice   makes   perfect "
assert answer_tokenize("  ab c  d") == ["ab", "c", "d"]
assert answer_left_extra("a  b c d e") == "a  b c d e"
assert answer_left_extra(" practice   makes   perfect") == "practice    makes   perfect"
assert answer_full_justify(["This", "is", "an"], 16) == "This    is    an"
assert answer_full_justify(["example", "of", "text"], 16) == "example  of text"
assert answer_full_justify(["hi"], 5) == "hi   "
for _ in range(300):
    t = random_text(rng)
    assert answer_reorder(t) == reorder_spaces_split(t)
    assert answer_tokenize(t) == t.split()
    assert answer_left_extra(t) == justify_left_extra(t)
print("Đáp án tham khảo: PASS toàn bộ!")

Đáp án tham khảo: PASS toàn bộ!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def reorderSpaces(self, text: str) -> str:
        spaces = text.count(" ")
        words = text.split()                 # split() KHÔNG tham số
        if len(words) == 1:                  # 0 khe -> dồn hết ra cuối
            return words[0] + " " * spaces
        gap, extra = divmod(spaces, len(words) - 1)   # chia cho SỐ KHE
        return (" " * gap).join(words) + " " * extra
```

**Câu thần chú:** *"Đếm cách, đếm khe, divmod, dư ra cuối — một từ thì dồn hết."*

**Checklist trước khi submit:**
- [ ] Dùng `split()` không tham số?
- [ ] Chia cho `len(words) - 1` (số khe), không phải `len(words)`?
- [ ] Đã xử lý 1 từ (tránh chia cho 0)?
- [ ] `len(kết quả) == len(text)`?
- [ ] Phần dư nằm ở **cuối**, không có dấu cách ở đầu?
- [ ] Tên method đúng `reorderSpaces`?